In [3]:
import os
import numpy as np
import json

folder= "/scratch/data/Camels_dataset_npy/"

for file in os.listdir(folder):
    print(os.path.join(folder,file))
    if file.endswith('.npy'):
        print(np.load(os.path.join(folder,file)).shape)


/scratch/data/Camels_dataset_npy/PUB_Feng
/scratch/data/Camels_dataset_npy/attr_671_name.json
/scratch/data/Camels_dataset_npy/attr_671.npy
(671, 35)
/scratch/data/Camels_dataset_npy/training_file
/scratch/data/Camels_dataset_npy/gageid.npy
(671,)
/scratch/data/Camels_dataset_npy/f_obs_19801001_20141001.npy
(671, 12418, 4)
/scratch/data/Camels_dataset_npy/f_obs_19801001_20141001_name.json


In [7]:
with open("/home/s0shar47/scratch/dPLHBV_PUB_Feng/PUBsplitLst.txt", "r") as f:
    data = json.load(f)
len(data)

10

In [8]:
import pandas as pd
pd.read_csv("/scratch/data/CAMELS_531.csv")

,HUC,Gauge ID,Gauge Name,Lat,Long,Area (km2)
0,1,1022500,narraguagus river at cherryfield maine,44.60797,-67.9352,573.60
1,1,1031500,piscataquis river near dover-foxcroft maine,45.17501,-69.3147,769.05
2,1,1047000,carrabassett river near north anson maine,44.86920,-69.9551,909.10
3,1,1052500,diamond river near wentworth location nh,44.87739,-71.0575,383.82
4,1,1054200,wild river at gilead maine,44.39044,-70.9796,180.98
...,...,...,...,...,...,...
526,18,11482500,redwood c a orick ca,41.29929,-124.0510,718.04
527,18,11522500,salmon r a somes bar ca,41.37763,-123.4780,1943.12
528,18,11523200,trinity r ab coffee c nr trinity ctr ca,41.11126,-122.7060,382.94
529,18,11528700,sf trinity r bl hyampom ca,40.64986,-123.4940,1980.08


In [24]:
### Feng's Fold Cleaned & Validated Setup

import os
import json
import numpy as np
import pandas as pd

# --------------------------------------------------
# Paths
# --------------------------------------------------
gageid_file = "/scratch/data/Camels_dataset_npy/gageid.npy"
attr_file = "/scratch/data/CAMELS_FORCING/daymet_19800101_20141231_attr_671.npy"
forcing_file = "/scratch/data/CAMELS_FORCING/daymet_19800101_20141231_forcing_671.npy"

pubsplit_file = "/home/s0shar47/scratch/dPLHBV_PUB_Feng/PUBsplitLst.txt"
camels531_file = "/scratch/data/CAMELS_531.csv"

out_root = "/home/s0shar47/scratch/Daymet_Folds"

# --------------------------------------------------
# Load data
# --------------------------------------------------
gageids = np.load(gageid_file)
attrs = np.load(attr_file)
forcings = np.load(forcing_file)

print("gageids shape:", gageids.shape)
print("attrs shape:", attrs.shape)
print("forcings shape:", forcings.shape)

with open(pubsplit_file, "r") as f:
    folds = json.load(f)

print("Number of folds:", len(folds))

# --------------------------------------------------
# Load 531 basin IDs & Pad Leading Zeros
# --------------------------------------------------
df531 = pd.read_csv(camels531_file, dtype={"Gauge ID": str}) 
test531_ids = set(df531["Gauge ID"].str.zfill(8))

print("531 basins:", len(test531_ids))

# --------------------------------------------------
# Standardize gageids array to 8-character strings
# --------------------------------------------------
gageids_str = np.array([str(int(float(x))).zfill(8) if str(x).replace('.', '').isdigit()
                        else str(x).zfill(8) for x in gageids])

# --------------------------------------------------
# Create folds
# --------------------------------------------------
for fold_num in range(10):

    fold_dir = os.path.join(out_root, str(fold_num + 1))
    train_dir = os.path.join(fold_dir, "train")
    test_dir = os.path.join(fold_dir, "test")

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)

    # ----------------------------------------------
    # Test basins = current fold ∩ CAMELS_531 (Ensure zfill)
    # ----------------------------------------------
    current_fold = set(str(x).zfill(8) for x in folds[fold_num])
    test_ids = current_fold.intersection(test531_ids)

    # ----------------------------------------------
    # Train basins = all other folds
    # ----------------------------------------------
    train_ids = set()
    for j in range(10):
        if j != fold_num:
            train_ids.update(str(x).zfill(8) for x in folds[j])

    # ----------------------------------------------
    # Find indices
    # ----------------------------------------------
    train_idx = np.where(np.isin(gageids_str, list(train_ids)))[0]
    test_idx = np.where(np.isin(gageids_str, list(test_ids)))[0]

    # ----------------------------------------------
    # Automated Alignment Check for ALL Basins
    # ----------------------------------------------
    # Train Check (Safe against float NaN values)
    train_forcing_ids_raw = forcings[train_idx, 0, 14]
    train_forcing_ids_str = np.array([
        str(int(float(x))).zfill(8) if not np.isnan(float(x)) else "NAN_ID" 
        for x in train_forcing_ids_raw
    ])
    expected_train_ids = gageids_str[train_idx]
    
    train_mismatches = np.where(train_forcing_ids_str != expected_train_ids)[0]
    
    if len(train_mismatches) == 0:
        print(f"✅ Fold {fold_num+1} Train Alignment Matrix: 100% PERFECT. All {len(train_idx)} basins match feature 14.")
    else:
        print(f"❌ WARNING: Fold {fold_num+1} Train Alignment FAILURE! {len(train_mismatches)} basins do not match.")
        for idx in train_mismatches[:3]:
            print(f"   Mismatch at index {idx}: Expected {expected_train_ids[idx]} but forcing feature 14 has {train_forcing_ids_str[idx]}")

    # Test Check (Safe against float NaN values)
    test_forcing_ids_raw = forcings[test_idx, 0, 14]
    test_forcing_ids_str = np.array([
        str(int(float(x))).zfill(8) if not np.isnan(float(x)) else "NAN_ID" 
        for x in test_forcing_ids_raw
    ])
    expected_test_ids = gageids_str[test_idx]
    
    test_mismatches = np.where(test_forcing_ids_str != expected_test_ids)[0]
    
    if len(test_mismatches) == 0:
        print(f"✅ Fold {fold_num+1} Test Alignment Matrix: 100% PERFECT. All {len(test_idx)} basins match feature 14.")
    else:
        print(f"❌ WARNING: Fold {fold_num+1} Test Alignment FAILURE! {len(test_mismatches)} basins do not match.")

    print(f"Fold {fold_num+1}: train={len(train_idx)} basins, test={len(test_idx)} basins\n")

    # ----------------------------------------------
    # # Save train datasets
    # # ----------------------------------------------
    # np.save(os.path.join(train_dir, "gageid.npy"), gageids[train_idx])
    # np.save(os.path.join(train_dir, "daymet_19800101_20141231_attr_671.npy"), attrs[train_idx])
    # np.save(os.path.join(train_dir, "daymet_19800101_20141231_forcing_671.npy"), forcings[train_idx])

    # # ----------------------------------------------
    # # Save test datasets
    # # ----------------------------------------------
    # np.save(os.path.join(test_dir, "gageid.npy"), gageids[test_idx])
    # np.save(os.path.join(test_dir, "daymet_19800101_20141231_attr_671.npy"), attrs[test_idx])
    # np.save(os.path.join(test_dir, "daymet_19800101_20141231_forcing_671.npy"), forcings[test_idx])

print("Done.")

gageids shape: (671,)
attrs shape: (671, 35)
forcings shape: (671, 12784, 15)
Number of folds: 10
531 basins: 531
✅ Fold 1 Train Alignment Matrix: 100% PERFECT. All 604 basins match feature 14.
✅ Fold 1 Test Alignment Matrix: 100% PERFECT. All 52 basins match feature 14.
Fold 1: train=604 basins, test=52 basins

✅ Fold 2 Train Alignment Matrix: 100% PERFECT. All 604 basins match feature 14.
✅ Fold 2 Test Alignment Matrix: 100% PERFECT. All 54 basins match feature 14.
Fold 2: train=604 basins, test=54 basins

✅ Fold 3 Train Alignment Matrix: 100% PERFECT. All 604 basins match feature 14.
✅ Fold 3 Test Alignment Matrix: 100% PERFECT. All 52 basins match feature 14.
Fold 3: train=604 basins, test=52 basins

✅ Fold 4 Train Alignment Matrix: 100% PERFECT. All 604 basins match feature 14.
✅ Fold 4 Test Alignment Matrix: 100% PERFECT. All 51 basins match feature 14.
Fold 4: train=604 basins, test=51 basins

✅ Fold 5 Train Alignment Matrix: 100% PERFECT. All 604 basins match feature 14.
✅ Fold

In [ ]:
import os
import shutil

root = "/home/s0shar47/scratch/Daymet_Folds"

attr_json = "/scratch/data/CAMELS_FORCING/daymet_19800101_20141231_attr_671_name.json"
forcing_json = "/scratch/data/CAMELS_FORCING/daymet_19800101_20141231_forcing_671_name.json"

for fold in range(1, 11):
    for split in ["train", "test"]:

        folder = os.path.join(root, str(fold), split)

        shutil.copy(
            attr_json,
            os.path.join(folder, "daymet_19800101_20141231_attr_671_name.json")
        )

        shutil.copy(
            forcing_json,
            os.path.join(folder, "daymet_19800101_20141231_forcing_671_name.json")
        )

print("JSON files copied to all train/test folders.")

JSON files copied to all train/test folders.


In [23]:
import os
import json
import pandas as pd
import numpy as np

pubsplit_file = "/home/s0shar47/scratch/dPLHBV_PUB_Feng/PUBsplitLst.txt"
forcing_file = "/scratch/data/CAMELS_FORCING/daymet_19800101_20141231_forcing_671.npy"

# 1. Load forcing data as a NumPy array
forcing = np.load(forcing_file)

# 2. Extract feature 14 as an array (Keep it as an array to count NaNs!)
g671_raw = forcing[:, 0, 14]
nan_count = np.isnan(g671_raw).sum()
print(f"Number of NaN values in g671: {nan_count}")

# 3. Standardize g671 to 8-character string IDs (Ignoring NaNs)
g671_set = set(
    str(int(float(x))).zfill(8) for x in g671_raw if not np.isnan(float(x))
)

# 4. Load the folds JSON file
with open(pubsplit_file, "r") as f:
    folds = json.load(f)

# 5. Gather all unique basin IDs across ALL 10 folds and pad them
all_fold_ids = set()
for fold in folds:
    all_fold_ids.update(str(x).zfill(8) for x in fold)

# 6. Find which unique values from folds are NOT present in g671
missing_from_g671 = all_fold_ids.difference(g671_set)

# --------------------------------------------------
# Output Results
# --------------------------------------------------
print(f"Unique basins in all folds combined: {len(all_fold_ids)}")
print(f"Unique valid basins found in g671: {len(g671_set)}")
print(f"Number of fold basins missing from g671: {len(missing_from_g671)}")

if len(missing_from_g671) > 0:
    print("\nThe missing gauge IDs are:")
    print(sorted(list(missing_from_g671)))

Number of NaN values in g671: 0
Unique basins in all folds combined: 671
Unique valid basins found in g671: 671
Number of fold basins missing from g671: 0


In [22]:
pd.read_csv("/scratch/data/CAMELS_FORCING/basin_dataset_public_v1p2/basin_mean_forcing/daymet/01/01195100_lump_cida_forcing_leap.txt",skiprows=3, sep='\s+')

,Year,Mnth,Day,Hr,dayl(s),prcp(mm/day),srad(W/m2),swe(mm),tmax(C),tmin(C),vp(Pa)
0,1980,1,1,12,32830.82,0.00,227.82,0.0,5.37,-5.04,433.64
1,1980,1,2,12,32832.00,0.00,200.04,0.0,4.58,-3.69,464.54
2,1980,1,3,12,32832.00,0.00,181.28,0.0,3.07,-4.05,449.69
3,1980,1,4,12,32832.00,0.00,217.11,0.0,-0.16,-9.22,301.80
4,1980,1,5,12,32832.00,2.39,141.28,0.0,-0.78,-6.42,366.18
...,...,...,...,...,...,...,...,...,...,...,...
12779,2014,12,27,12,32486.40,0.00,217.96,0.0,9.83,-1.00,560.00
12780,2014,12,28,12,32486.40,1.00,205.93,0.0,11.00,1.28,664.56
12781,2014,12,29,12,32486.40,0.00,200.68,0.0,8.38,-0.50,599.86
12782,2014,12,30,12,32486.40,0.00,202.82,0.0,3.11,-5.50,400.00


In [7]:
import numpy as np

gw= np.load("/home/s0shar47/scratch/DAYMET_BASEFLOW_OUTPUTS/1/LSTM_HBV_E1_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/gwflow.npy")
ob= np.load("/home/s0shar47/scratch/DAYMET_BASEFLOW_OUTPUTS/1/LSTM_HBV_E1_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/baseflow.npy")

In [8]:
gw= gw.squeeze()
gw.shape

(3286, 52)

In [6]:
import os
methods= ['Furey', 'USGS', 'Eckhardt', 'LocalHYSEP']
models= ['HBV','HBV_capillary', 'marrmot_PRMS']
losses= ['RmseLoss_flow_baseflow','RmseLoss1_flow_baseflow', 'NseLoss_flow_baseflow']
weights= ['2575','5050','7525']

for method in methods:
    for model in models:
        for loss in losses:
            for weight in weights:
                try:
                    os.listdir(f"home/s0sha47/scratch/salil/DPL/CASES/{method}/{model}/{loss}/{weight}/OUTPUTS")
                except:
                    print(f"{method}/{model}/{loss}/{weight} does not exists")

Furey/HBV/RmseLoss_flow_baseflow/2575 does not exists
Furey/HBV/RmseLoss_flow_baseflow/5050 does not exists
Furey/HBV/RmseLoss_flow_baseflow/7525 does not exists
Furey/HBV/RmseLoss1_flow_baseflow/2575 does not exists
Furey/HBV/RmseLoss1_flow_baseflow/5050 does not exists
Furey/HBV/RmseLoss1_flow_baseflow/7525 does not exists
Furey/HBV/NseLoss_flow_baseflow/2575 does not exists
Furey/HBV/NseLoss_flow_baseflow/5050 does not exists
Furey/HBV/NseLoss_flow_baseflow/7525 does not exists
Furey/HBV_capillary/RmseLoss_flow_baseflow/2575 does not exists
Furey/HBV_capillary/RmseLoss_flow_baseflow/5050 does not exists
Furey/HBV_capillary/RmseLoss_flow_baseflow/7525 does not exists
Furey/HBV_capillary/RmseLoss1_flow_baseflow/2575 does not exists
Furey/HBV_capillary/RmseLoss1_flow_baseflow/5050 does not exists
Furey/HBV_capillary/RmseLoss1_flow_baseflow/7525 does not exists
Furey/HBV_capillary/NseLoss_flow_baseflow/2575 does not exists
Furey/HBV_capillary/NseLoss_flow_baseflow/5050 does not exists
F

In [8]:
import os

methods = ['Furey', 'USGS', 'Eckhardt', 'LocalHYSEP']
models = ['HBV', 'HBV_capillary', 'marrmot_PRMS']
losses = ['RmseLoss_flow_baseflow', 'RmseLoss1_flow_baseflow', 'NseLoss_flow_baseflow']
weights = ['2575', '5050', '7525']

missing_dirs = []

for method in methods:
    for model in models:
        for loss in losses:
            for weight in weights:
                # Added a leading slash or ensure the path is correct relative to your working directory
                path = f"/home/s0shar47/scratch/salil/DPL/CASES/{method}/{model}/{loss}/{weight}/OUTPUTS"
                if not os.path.isdir(path):
                    missing_dirs.append(f"{method}/{model}/{loss}/{weight}")
                    print(f"{method}/{model}/{loss}/{weight} does not exist")
# "Z:\salil\DPL\CASES\Eckhardt\HBV\NseLoss_flow_baseflow\2575\OUTPUTS"
print(f"\nTotal missing directories: {len(missing_dirs)}")

USGS/HBV/RmseLoss_flow_baseflow/2575 does not exist
USGS/HBV/RmseLoss_flow_baseflow/5050 does not exist
USGS/HBV/RmseLoss_flow_baseflow/7525 does not exist
USGS/HBV/RmseLoss1_flow_baseflow/2575 does not exist
USGS/HBV/RmseLoss1_flow_baseflow/5050 does not exist
USGS/HBV/RmseLoss1_flow_baseflow/7525 does not exist
USGS/HBV/NseLoss_flow_baseflow/2575 does not exist
USGS/HBV/NseLoss_flow_baseflow/5050 does not exist
USGS/HBV/NseLoss_flow_baseflow/7525 does not exist
USGS/HBV_capillary/RmseLoss_flow_baseflow/2575 does not exist
USGS/HBV_capillary/RmseLoss_flow_baseflow/5050 does not exist
USGS/HBV_capillary/RmseLoss_flow_baseflow/7525 does not exist
USGS/HBV_capillary/RmseLoss1_flow_baseflow/2575 does not exist
USGS/HBV_capillary/RmseLoss1_flow_baseflow/5050 does not exist
USGS/HBV_capillary/RmseLoss1_flow_baseflow/7525 does not exist
USGS/HBV_capillary/NseLoss_flow_baseflow/2575 does not exist
USGS/HBV_capillary/NseLoss_flow_baseflow/5050 does not exist
USGS/HBV_capillary/NseLoss_flow_bas

In [3]:
## Select the number of objectives
# objective= 'Multi'
objectives= ['Single', 'Multi']

method= 'Furey'
methods= ['USGS', 'Furey', 'Eckhardt', 'LocalHYSEP']

for method in methods:
    ## Select the test
    test= 'PUR'
    tests= ['PUB','PUR']

    ## Select the model
    model= 'HBV'
    models= ['HBV', 'PRMS', 'HBV_capillary']

    if model=='HBV':
        dyn= 'parBETA_parBETAET_'

    if model=='marrmot_PRMS':
        dyn= 'alpha_cgw_scx_tt_'

    if model=='HBV_capillary':
        dyn= 'parBETA_parBETAET_parK0_parUZL_'

    if test== 'PUB':
        n=11
    if test== 'PUR':
        n=8

        
    import numpy as np
    import os

    def calc_nse(obs, sim):
        mask = np.isfinite(obs) & np.isfinite(sim)
        obs = obs[mask]
        sim = sim[mask]

        if len(obs) < 2:
            return np.nan

        den = np.sum((obs - np.mean(obs)) ** 2)
        if den == 0:
            return np.nan

        return 1 - np.sum((obs - sim) ** 2) / den


    def calc_kge(obs, sim):
        mask = np.isfinite(obs) & np.isfinite(sim)
        obs = obs[mask]
        sim = sim[mask]

        if len(obs) < 2:
            return np.nan

        r = np.corrcoef(obs, sim)[0, 1]

        std_obs = np.std(obs)
        std_sim = np.std(sim)

        if std_obs == 0:
            return np.nan

        alpha = std_sim / std_obs

        mean_obs = np.mean(obs)
        if mean_obs == 0:
            return np.nan

        beta = np.mean(sim) / mean_obs

        return 1 - np.sqrt(
            (r - 1) ** 2 +
            (alpha - 1) ** 2 +
            (beta - 1) ** 2
        )


    def calc_bias(obs, sim):
        mask = np.isfinite(obs) & np.isfinite(sim)
        obs = obs[mask]
        sim = sim[mask]

        den = np.sum(obs)
        if den == 0:
            return np.nan

        return (np.sum(sim) - den) / den


    def calc_abs_bias(obs, sim):
        mask = np.isfinite(obs) & np.isfinite(sim)
        obs = obs[mask]
        sim = sim[mask]

        den = np.sum(obs)
        if den == 0:
            return np.nan

        return np.abs(np.sum(sim) - den) / den


    models= ['HBV', 'HBV_capillary', 'marrmot_PRMS']
    for k in range(len(models)):
        model= models[k]

        if model=='HBV':
            dyn= 'parBETA_parBETAET_'

        if model=='marrmot_PRMS':
            dyn= 'alpha_cgw_scx_tt_'

        if model=='HBV_capillary':
            dyn= 'parBETA_parBETAET_parK0_parUZL_'

    # ============================================================
    # Metric functions
    # ============================================================



    # ============================================================
    # Containers
    # ============================================================

        metrics_old = {
            "NSE": [],
            "KGE": [],
            "Bias": [],
            "AbsBias": [],
            "NSE_low": [],
            "NSE_high": [],
            "Bias_low": [],
            "Bias_high": [],
            "AbsBias_low": [],
            "AbsBias_high": [],
        }

        metrics_new = {
            "NSE": [],
            "KGE": [],
            "Bias": [],
            "AbsBias": [],
            "NSE_low": [],
            "NSE_high": [],
            "Bias_low": [],
            "Bias_high": [],
            "AbsBias_low": [],
            "AbsBias_high": [],
        }

        # ============================================================
        # Loop over folds
        # ============================================================

        for i in range(1, n):


            if test== 'PUB':
                folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/{method}_{objectives[1]}/{i}/"
                    
                sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUB/OUTPUTS_PUB_{objectives[0]}/{i}/"
    

            if test== 'PUR':
                folder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/{method}_{objectives[1]}/Region_{i}/"
                sfolder= f"/home/s0shar47/scratch/OUTPUTS_Paper1/PUR/OUTPUTS_PUR_{objectives[0]}/Region_{i}/"


            obs_flow = np.load(
                os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/baseflow.npy")
            )

            sim_old = np.load(
                os.path.join(sfolder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy")
                # f"/home/s0shar47/scratch/OUTPUTS_PUR_Single/Region_{i}/LSTM_HBV_capillary_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_parK0_parUZL_/ts1989_1999/flow_sim.npy"

            )

            sim_new = np.load(
                os.path.join(folder,f"LSTM_{model}_E50_R365_B100_H256_tr1989_1999_n16_111111/{dyn}/ts1989_1999/gwflow.npy")
                # f"/home/s0shar47/scratch/OUTPUTS_PUR_Multi/Region_{i}/LSTM_HBV_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_/ts1989_1999/flow_sim.npy"
                # f"/home/s0shar47/scratch/OUTPUTS_PUR_Multi/Region_{i}/LSTM_HBV_capillary_E50_R365_B100_H256_tr1989_1999_n16_111111/parBETA_parBETAET_parK0_parUZL_/ts1989_1999/flow_sim.npy"
            )

            sim_old = np.squeeze(sim_old)
            sim_new = np.squeeze(sim_new)

            # --------------------------------------------------------
            # Loop over basins
            # --------------------------------------------------------
            for k in range(obs_flow.shape[1]):

                obs = obs_flow[:, k]
                old = sim_old[:, k]
                new = sim_new[:, k]

                # =====================================================
                # Overall metrics
                # =====================================================
                metrics_old["NSE"].append(calc_nse(obs, old))
                metrics_new["NSE"].append(calc_nse(obs, new))

                metrics_old["KGE"].append(calc_kge(obs, old))
                metrics_new["KGE"].append(calc_kge(obs, new))

                metrics_old["Bias"].append(calc_bias(obs, old))
                metrics_new["Bias"].append(calc_bias(obs, new))

                metrics_old["AbsBias"].append(calc_abs_bias(obs, old))
                metrics_new["AbsBias"].append(calc_abs_bias(obs, new))

                # =====================================================
                # Low / High flow
                # =====================================================
                threshold = np.median(obs)

                low = obs <= threshold
                high = obs > threshold

                # ---------------- Low flow ----------------
                if np.sum(low) > 2:

                    metrics_old["NSE_low"].append(calc_nse(obs[low], old[low]))
                    metrics_new["NSE_low"].append(calc_nse(obs[low], new[low]))

                    metrics_old["Bias_low"].append(calc_bias(obs[low], old[low]))
                    metrics_new["Bias_low"].append(calc_bias(obs[low], new[low]))

                    metrics_old["AbsBias_low"].append(
                        calc_abs_bias(obs[low], old[low])
                    )
                    metrics_new["AbsBias_low"].append(
                        calc_abs_bias(obs[low], new[low])
                    )

                # ---------------- High flow ----------------
                if np.sum(high) > 2:

                    metrics_old["NSE_high"].append(calc_nse(obs[high], old[high]))
                    metrics_new["NSE_high"].append(calc_nse(obs[high], new[high]))

                    metrics_old["Bias_high"].append(calc_bias(obs[high], old[high]))
                    metrics_new["Bias_high"].append(calc_bias(obs[high], new[high]))

                    metrics_old["AbsBias_high"].append(
                        calc_abs_bias(obs[high], old[high])
                    )
                    metrics_new["AbsBias_high"].append(
                        calc_abs_bias(obs[high], new[high])
                    )

        # ============================================================
        # Print comparison
        # ============================================================

        # print("\n" + "=" * 60)
        print(f"{test} Test BASEFLOW COMPARISON ({model} 1989-1999) using {method} Baseflow Separation")
        # print("=" * 60)

        metric_order = [
            "NSE",
            "KGE",
            "Bias",
            "AbsBias",
            "NSE_low",
            "NSE_high",
            "Bias_low",
            "Bias_high",
            "AbsBias_low",
            "AbsBias_high",
        ]

        for metric in metric_order:
            old = np.nanmedian(metrics_old[metric])
            new = np.nanmedian(metrics_new[metric])
            print(f"{metric:<13}: {old:8.4f} → {new:8.4f}")

        print("=" * 60)


PUR Test BASEFLOW COMPARISON (HBV 1989-1999) using USGS Baseflow Separation
NSE          :  -0.3849 →   0.3813
KGE          :   0.1078 →   0.4475
Bias         :   0.4531 →   0.0299
AbsBias      :   0.4965 →   0.3010
NSE_low      :  -7.3397 →  -2.9620
NSE_high     :  -2.4261 →  -0.2832
Bias_low     :   0.9043 →   0.4030
Bias_high    :   0.4019 →  -0.0443
AbsBias_low  :   0.9056 →   0.5698
AbsBias_high :   0.4674 →   0.3042
PUR Test BASEFLOW COMPARISON (HBV_capillary 1989-1999) using USGS Baseflow Separation
NSE          :   0.0889 →   0.3369
KGE          :   0.2640 →   0.3998
Bias         :   0.1220 →   0.0058
AbsBias      :   0.4514 →   0.3250
NSE_low      :  -5.0612 →  -3.1366
NSE_high     :  -0.9202 →  -0.4044
Bias_low     :   0.5830 →   0.4042
Bias_high    :   0.0291 →  -0.0617
AbsBias_low  :   0.7429 →   0.5790
AbsBias_high :   0.4120 →   0.3072
PUR Test BASEFLOW COMPARISON (marrmot_PRMS 1989-1999) using USGS Baseflow Separation
NSE          :  -0.8360 →   0.3801
KGE          :   0

/home/s0shar47/.local/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/s0shar47/.local/lib/python3.10/site-packages/numpy/lib/_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


PUR Test BASEFLOW COMPARISON (HBV 1989-1999) using LocalHYSEP Baseflow Separation
NSE          :   0.3942 →   0.5294
KGE          :   0.4046 →   0.5513
Bias         :  -0.0556 →  -0.0011
AbsBias      :   0.2839 →   0.2018
NSE_low      :  -4.1296 →  -4.1461
NSE_high     :   0.1343 →   0.2655
Bias_low     :   0.4178 →   0.3563
Bias_high    :  -0.1219 →  -0.0590
AbsBias_low  :   0.5631 →   0.4942
AbsBias_high :   0.2745 →   0.1939
PUR Test BASEFLOW COMPARISON (HBV_capillary 1989-1999) using LocalHYSEP Baseflow Separation
NSE          :   0.2679 →   0.5104
KGE          :   0.2429 →   0.5287
Bias         :  -0.3001 →   0.0049
AbsBias      :   0.4094 →   0.2342
NSE_low      :  -2.7490 →  -3.9562
NSE_high     :  -0.0865 →   0.2662
Bias_low     :   0.1972 →   0.3513
Bias_high    :  -0.3613 →  -0.0631
AbsBias_low  :   0.5376 →   0.5041
AbsBias_high :   0.4273 →   0.2049
PUR Test BASEFLOW COMPARISON (marrmot_PRMS 1989-1999) using LocalHYSEP Baseflow Separation
NSE          :   0.3400 →   0.5063
